# 02. Gurobi로 원래 MILP 해결

Gurobi 결과는 SA/QA 비교의 **true optimum reference**로 사용한다.

세 가지 모형을 푼다.

| 모형 | 변수 | 의미 |
|---|---|---|
| `SS` | `x_ij in {0,1}` | Single-source |
| `MS` | `q_ij` 연속 | 원래 Multiple-source (x_ij in [0,1]과 동일) |
| `MS_INT` | `q_ij` 정수 | QUBO와 동일한 1 unit 이산화 수준 |

`MS`와 `MS_INT`를 모두 기록하면, QUBO 기반 해의 gap에서 **이산화로 인한 손실**과 **solver 품질로 인한 손실**을 분리할 수 있다.

linking 제약 `x_ij <= y_j`는 capacity 제약에 의해 함의되는 redundant 제약이므로 SS/MS/QUBO 모두에서 제거하였다(동일한 feasible set).

In [ ]:
# 프로젝트 루트를 import 경로에 추가한다.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config, resolve_path

config = load_config(PROJECT_ROOT / "config" / "experiment_config.yaml")
DATA_DIR = resolve_path(config, "data_dir")
RAW_DIR = resolve_path(config, "raw_dir")
PROCESSED_DIR = resolve_path(config, "processed_dir")
FIGURE_DIR = resolve_path(config, "figure_dir")
SOLUTION_DIR = RAW_DIR / "solutions"
SOLUTION_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("설정 로드 완료:", len(config["instances"]), "개 instance")


In [ ]:
from src import cflp_ms, cflp_ss, gurobi_solver
from src.data_generator import CFLPInstance
from src.persistence import save_solution, save_table

instances = [
    CFLPInstance.load(DATA_DIR / f"{spec['name']}.json")
    for spec in config["instances"]
]
records = []
for instance in instances:
    references = gurobi_solver.solve_all(instance, config["gurobi"])
    for key, result in references.items():
        save_solution(result.solution, SOLUTION_DIR, instance.name, f"gurobi_{key}")
        record = {"instance": instance.name, "size": instance.num_customers}
        record.update(result.to_record())
        records.append(record)
    print(
        f"{instance.name}: SS={references['SS'].objective:.4f}, "
        f"MS={references['MS'].objective:.4f}, "
        f"MS_INT={references['MS_INT'].objective:.4f}"
    )

gurobi_results = pd.DataFrame(records)
save_table(gurobi_results, RAW_DIR, "gurobi_results.csv")
gurobi_results

## 최적성 확인

모든 모형이 `OPTIMAL` 상태이고 MIP gap이 0인지 확인한다.

In [ ]:
all_optimal = bool((gurobi_results["status"] == "OPTIMAL").all())
max_gap = float(gurobi_results["mip_gap"].max())
print("모두 OPTIMAL:", all_optimal)
print("최대 MIP gap:", max_gap)
if not all_optimal:
    raise RuntimeError("OPTIMAL이 아닌 모형이 있습니다. 결과를 reference로 쓸 수 없습니다.")

## 해의 feasibility 재검증과 대소 관계

SS는 MS의 해를 이진 배정으로 제한한 문제이므로 다음 관계가 성립해야 한다.

$$Obj_{MS} \le Obj_{MS\text{-}int} \le Obj_{SS}$$

이 관계가 깨지면 formulation 구현에 오류가 있다는 뜻이다.

In [ ]:
tolerance = float(config["feasibility"]["tolerance"])
checks = []
for instance in instances:
    subset = gurobi_results[gurobi_results["instance"] == instance.name]
    values = dict(zip(subset["gurobi_model"], subset["objective"]))
    checks.append(
        {
            "instance": instance.name,
            "MS": round(values["MS"], 4),
            "MS_INT": round(values["MS-int-q"], 4),
            "SS": round(values["SS"], 4),
            "ordering_ok": bool(
                values["MS"] <= values["MS-int-q"] + 1e-6
                and values["MS-int-q"] <= values["SS"] + 1e-6
            ),
            "discretization_loss": round(values["MS-int-q"] - values["MS"], 6),
            "single_source_premium": round(values["SS"] - values["MS"], 4),
        }
    )
ordering = pd.DataFrame(checks)
save_table(ordering, RAW_DIR, "gurobi_ordering_check.csv")
ordering